# 🛣️ Real-Time Pothole Segmentation & Road Damage Assessment with YOLOv8

This notebook documents the complete fine-tuning, evaluation, and deployment workflow for pothole instance segmentation using **Ultralytics YOLOv8-seg** on NVIDIA GPU.

### 📌 Pipeline Structure:
1. **Environment Setup & GPU Check** (NVIDIA RTX 2050 CUDA)
2. **Dataset Configuration & Inspection** (`data.yaml`)
3. **Baseline Model Validation** (`models/pothole_v2_training.pt`)
4. **Fine-Tuning YOLOv8-Seg** with Augmentations
5. **Final Model Evaluation & Metrics** (mAP50, mAP50-95, Mask Precision/Recall)
6. **Road Damage Severity Assessment Inference**
7. **Video Inference with ByteTrack & Road ROI Optimization**
8. **Multi-Backend Model Export** (ONNX / TensorRT)

In [ ]:
# 1. GPU & Environment Verification
import torch
import ultralytics
from ultralytics import YOLO

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
    print("GPU VRAM:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")
print("Ultralytics Version:", ultralytics.__version__)

In [ ]:
# 2. Inspect data.yaml & Dataset Splits
import os
import yaml

with open("data.yaml", "r") as f:
    cfg = yaml.safe_load(f)
print("data.yaml:", cfg)

train_len = len(os.listdir("dataset/train/images"))
val_len = len(os.listdir("dataset/valid/images"))
print(f"Dataset: {train_len} training images, {val_len} validation images")

In [ ]:
# 3. Establish Baseline Validation Metrics
baseline_model = YOLO("models/pothole_v2_training.pt")
base_metrics = baseline_model.val(data="data.yaml", imgsz=640, device=0, split="val")
print("Baseline Mask mAP50:", base_metrics.seg.map50)
print("Baseline Mask mAP50-95:", base_metrics.seg.map)
print("Baseline Speed (ms):", base_metrics.speed)

In [ ]:
# 4. Fine-Tune Model on GPU
train_results = baseline_model.train(
    data="data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project="runs/pothole",
    name="pothole_v2_training",
    exist_ok=True,
    mosaic=1.0,
    mixup=0.1,
    close_mosaic=10,
    save=True
)

In [ ]:
# 5. Validate Fine-Tuned Model
final_model = YOLO("models/pothole_v2_final.pt")
final_metrics = final_model.val(data="data.yaml", imgsz=640, device=0)
print("Final Mask mAP50:", final_metrics.seg.map50)
print("Final Mask mAP50-95:", final_metrics.seg.map)

In [ ]:
# 6. Test Inference on Sample Image
from scripts.inference import run_inference
summary, annotated = run_inference(
    source="output_preview_1.jpg",
    model_path="models/pothole_v2_final.pt",
    imgsz=640,
    conf=0.25
)
print("Severity Grade:", summary["damage_severity_label"])
print("Potholes Detected:", summary["potholes_count"])

In [ ]:
# 7. Export Model to ONNX and TensorRT
from scripts.export import export_models
export_models(weights_path="models/pothole_v2_final.pt", imgsz=640)